In [ ]:
import numpy as np 
import pandas as pd 
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.utils import make_grid
import matplotlib.pyplot as plt 


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.2, contrast=0.1),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
DATA_DIR = "/kaggle/input/datasets/asdasdasasdas/garbage-classification/Garbage classification/Garbage classification"  # Path to unzipped dataset folder
train_dataset = datasets.ImageFolder(root=DATA_DIR, transform=train_transform)
test_dataset = datasets.ImageFolder(root=DATA_DIR, transform=test_transform)
Classes = train_dataset.classes
num_classes = len(Classes)
n = len(train_dataset)
indices = np.random.permutation(n)
split = int(0.8 * n)
train_inx, test_inx = indices[:split], indices[split:]
train_subset = Subset(train_dataset, train_inx)
test_subset = Subset(test_dataset, test_inx)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)



epochs = 10
best_acc = 0

for epoch in range(epochs):
    print(f"epoch: {epoch + 1} / {epochs}")
    print("-" * 25)

    model.train()
    running_loss, running_corr = 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, pred = torch.max(outputs, 1)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corr += torch.sum(pred == labels.data)

    epoch_loss = running_loss / len(train_subset)
    epoch_acc = running_corr.double() / len(train_subset)
    print(f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f}")

    model.eval()

    val_loss, val_corr = 0, 0
    

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, pred = torch.max(outputs, 1)

            val_loss += loss.item() * inputs.size(0)
            val_corr += torch.sum(pred == labels.data)

        val_loss = val_loss / len(test_subset)
        val_acc = val_corr.double() / len(test_subset)
        print(f"val Loss: {val_loss:.4f} | val Acc: {val_acc:.4f}")

        if val_acc > best_acc:
                best_acc = val_acc
                torch.save(model.state_dict(), "waste_classifier_resnet18.pth")
                print("--> Saved best model checkpoint.")
        
    print(f"\nTraining Complete. Best Validation Accuracy: {best_acc:.4f}")